# Dunnhumby M5 가치 정밀도 진단
기존 seed-42 M1 체크포인트로 (1) 거래 횟수로 축소추정한 거래당 가치와 원래 값 중 무엇이 신규상품 가격위치에 더 맞는지, (2) 가격 적합이 CLV 구간에 따라 정말 달라지는지, (3) 첫 구매 학습행 비중을 확인합니다. 재학습·최종 test·holdout은 수행하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '42c13f2'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip().startswith(REVIEWED_SHA)


In [ ]:
import importlib
import json
import torch
import lightgcn_clv_m5_value_precision_diagnostic as precision
precision = importlib.reload(precision)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
assert precision.CODE_VERSION == 'm5-value-precision-diagnostic-v1'
cfg = precision.configure_value_precision_diagnostic('dunnhumby')
print(json.dumps(precision.preflight_summary(cfg), ensure_ascii=False, indent=2))


In [ ]:
precision = importlib.reload(precision)
cfg = precision.configure_value_precision_diagnostic('dunnhumby')
paths = precision.run_value_precision_diagnostic(cfg)


In [ ]:
import json
import pandas as pd
from IPython.display import display

summary = pd.read_csv(paths['summary_csv'])
print('1) 누락 정답이 M1 Top-10 오추천을 이기는 비율')
display(summary)
report = json.load(open(paths['json']))
print('2) 사용자 단위 bootstrap 95% 구간')
display(pd.DataFrame(report['bootstrap']['contrasts']).T)
print('3) 첫 구매 학습행 비중')
print(json.dumps(report['first_purchase_rows'], ensure_ascii=False, indent=2))
print('판독: 축소추정은 두 데이터 모두에서 원래 값 이상이고 한 데이터 이상에서 구간이 0을 제외할 때만 채택합니다.')
print('결과 파일:', paths)
